# PySpark across Polaris, Hive, HDFS, Iceberg, and SeaweedFS S3

Start with **17-0-Jupyter.md**. Run this notebook in the Jupyter container, from top to bottom, using a fresh Python 3 kernel. Close another Spark kernel first: this lab has one worker.

Lesson **16-polaris.md** must leave `demo_catalog_temp.demo_database_temp.demo_table_temp` with three rows and its snapshot history. This notebook reads that retained table. It never drops or changes the lesson 16 table.

Every execution creates a new namespace and storage prefix ending in `temp`. New examples remain available after Spark stops; no cleanup cells run automatically. Run All creates a new independent set of examples on a fresh kernel. Re-running an individual CREATE cell in the same run can report that a resource already exists; inspect it rather than overwriting it.


## 1. Imports, home directory, and a run name

The notebook kernel lives in Kubernetes, not on the VM host. Its home is `/workspace`, backed by the `spark-notebooks` PVC. Use `/workspace/notebooks` as the only notebook working directory. Open and save this notebook directly in JupyterLab; no VM-home copy is needed.

Create the run name once. Keep it for the rest of this notebook.


In [ ]:
import os
import json
import socket
import uuid
from pathlib import Path
from datetime import datetime, timezone
import requests

notebook_home = Path('/workspace/notebooks')
notebook_home.mkdir(parents=True, exist_ok=True)
run_id = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:8] + '_temp'
namespace = 'notebook_demo_' + run_id
print('Kernel home:', Path.home())
print('Notebook directory:', notebook_home)
print('Current working directory:', Path.cwd())
print('Run namespace:', namespace)


## 2. Resolve cluster connections

Service names are reachable from the kernel and executors. VM port-forwards are for the browser only. The deployment's generic `HDFS_URI` can be `file:///`, so this lesson explicitly selects the in-cluster NameNode with `NOTEBOOK_HDFS_URI` as an optional override.


In [ ]:
master_url = os.environ.get('SPARK_MASTER_URL', 'spark://spark-master:7077')
hdfs_uri = os.environ.get('NOTEBOOK_HDFS_URI', 'hdfs://namenode:9000').rstrip('/')
metastore_uri = 'thrift://hive-metastore:9083'
polaris_uri = os.environ.get('POLARIS_URI', 'http://polaris:8181/api/catalog').rstrip('/')
polaris_base = polaris_uri.removesuffix('/api/catalog')
s3_endpoint = os.environ.get('S3_ENDPOINT', 'http://seaweedfs:8333')
hdfs_root = hdfs_uri + '/training/notebook-demo/' + run_id
s3_root = 's3://demo-catalog-temp/iceberg/' + namespace
s3a_root = s3_root.replace('s3://', 's3a://', 1)
for name, value in [('Spark master', master_url), ('HDFS', hdfs_uri), ('Hive Metastore', metastore_uri), ('Polaris', polaris_uri), ('S3 endpoint', s3_endpoint), ('HDFS run directory', hdfs_root), ('S3 run prefix', s3_root)]:
    print(f'{name}: {value}')


Read credentials from the existing container environment without printing them. Polaris OAuth credentials authorize catalog requests; S3 keys authorize object operations.


In [ ]:
required = ['POLARIS_CLIENT_ID', 'POLARIS_CLIENT_SECRET', 'AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY']
missing = [name for name in required if not os.environ.get(name)]
assert not missing, 'Missing container environment variables: ' + ', '.join(missing)
polaris_credential = os.environ['POLARIS_CLIENT_ID'] + ':' + os.environ['POLARIS_CLIENT_SECRET']
s3_access_key = os.environ['AWS_ACCESS_KEY_ID']
s3_secret_key = os.environ['AWS_SECRET_ACCESS_KEY']
print('Required credential variables are present; values are not displayed.')


## 3. Verify lesson 16 before creating anything

Obtain a short-lived OAuth token and load the existing table registration. If this cell reports 404, complete lesson 16 and keep its data. It does not silently recreate or replace that prerequisite.


In [ ]:
token_response = requests.post(
    polaris_uri + '/v1/oauth/tokens',
    headers={'Polaris-Realm': 'POLARIS'},
    data={'grant_type': 'client_credentials', 'client_id': os.environ['POLARIS_CLIENT_ID'],
          'client_secret': os.environ['POLARIS_CLIENT_SECRET'], 'scope': 'PRINCIPAL_ROLE:ALL'},
    timeout=30)
token_response.raise_for_status()
api_headers = {'Authorization': 'Bearer ' + token_response.json()['access_token'], 'Polaris-Realm': 'POLARIS'}
baseline_path = '/v1/demo_catalog_temp/namespaces/demo_database_temp/tables/demo_table_temp'
baseline_response = requests.get(polaris_uri + baseline_path, headers=api_headers, timeout=30)
assert baseline_response.status_code != 404, 'Complete lesson 16 first; its catalog, namespace, table, and S3 bucket must be retained.'
baseline_response.raise_for_status()
baseline_metadata = baseline_response.json()
assert baseline_metadata['metadata-location'].startswith('s3://demo-catalog-temp/iceberg/')
print('Existing Polaris table UUID:', baseline_metadata['metadata']['table-uuid'])
print('Existing metadata location:', baseline_metadata['metadata-location'])


## 4. Configure Spark and all catalog backends

The kernel is the Spark driver and submits work to the standalone master. One executor core and 512 MiB executor memory suit this small lab. Driver event logging is disabled so this exercise does not write history files to the shared `warehouse` bucket.

Start from a fresh kernel: catalog configuration is established before creating Spark. This explicitly replaces the image's Delta session catalog with Iceberg's Hive-aware session catalog.

| Spark catalog | Backend | File storage |
|---|---|---|
| `spark_catalog` | Hive Metastore, including ordinary Hive tables | HDFS |
| `hadoop_iceberg` | Iceberg HadoopCatalog | HDFS |
| `hive_iceberg` | Iceberg HiveCatalog | HDFS |
| `demo_catalog_temp` | Polaris Iceberg REST catalog | SeaweedFS S3 |


In [ ]:
os.environ['PYSPARK_SUBMIT_ARGS'] = '--driver-memory 512m pyspark-shell'
from pyspark.sql import SparkSession

builder = (SparkSession.builder.appName('17-1-pyspark-' + run_id)
    .master(master_url)
    .config('spark.driver.host', os.environ.get('POD_IP') or socket.gethostbyname(socket.gethostname()))
    .config('spark.driver.bindAddress', '0.0.0.0')
    .config('spark.cores.max', '1')
    .config('spark.executor.cores', '1')
    .config('spark.executor.memory', '512m')
    .config('spark.sql.shuffle.partitions', '1')
    .config('spark.eventLog.enabled', 'false')
    .config('spark.sql.session.timeZone', 'UTC')
    .config('spark.hadoop.fs.defaultFS', hdfs_uri)
    .config('spark.hadoop.dfs.client.use.datanode.hostname', 'true')
    .config('spark.hadoop.hive.metastore.uris', metastore_uri)
    .config('spark.sql.warehouse.dir', hdfs_root + '/hive-warehouse')
    .config('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.iceberg.spark.SparkSessionCatalog')
    .config('spark.sql.catalog.spark_catalog.type', 'hive')
    .config('spark.sql.catalog.spark_catalog.uri', metastore_uri)
    .config('spark.sql.catalog.hadoop_iceberg', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.hadoop_iceberg.type', 'hadoop')
    .config('spark.sql.catalog.hadoop_iceberg.warehouse', hdfs_root + '/hadoop-warehouse')
    .config('spark.sql.catalog.hive_iceberg', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.hive_iceberg.type', 'hive')
    .config('spark.sql.catalog.hive_iceberg.uri', metastore_uri)
    .config('spark.sql.catalog.hive_iceberg.warehouse', hdfs_root + '/hive-iceberg-warehouse')
    .enableHiveSupport())


Configure the Polaris alias to select the retained catalog, and configure Iceberg S3FileIO separately. The `warehouse` value here is a Polaris catalog name, not an S3 URI.


In [ ]:
builder = (builder
    .config('spark.sql.catalog.demo_catalog_temp', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.demo_catalog_temp.type', 'rest')
    .config('spark.sql.catalog.demo_catalog_temp.uri', polaris_uri)
    .config('spark.sql.catalog.demo_catalog_temp.warehouse', 'demo_catalog_temp')
    .config('spark.sql.catalog.demo_catalog_temp.oauth2-server-uri', polaris_uri + '/v1/oauth/tokens')
    .config('spark.sql.catalog.demo_catalog_temp.credential', polaris_credential)
    .config('spark.sql.catalog.demo_catalog_temp.scope', 'PRINCIPAL_ROLE:ALL')
    .config('spark.sql.catalog.demo_catalog_temp.rest.auth.type', 'oauth2')
    .config('spark.sql.catalog.demo_catalog_temp.header.Polaris-Realm', 'POLARIS')
    .config('spark.sql.catalog.demo_catalog_temp.rest-metrics-reporting-enabled', 'false')
    .config('spark.sql.catalog.demo_catalog_temp.io-impl', 'org.apache.iceberg.aws.s3.S3FileIO')
    .config('spark.sql.catalog.demo_catalog_temp.s3.endpoint', s3_endpoint)
    .config('spark.sql.catalog.demo_catalog_temp.s3.path-style-access', 'true')
    .config('spark.sql.catalog.demo_catalog_temp.s3.access-key-id', s3_access_key)
    .config('spark.sql.catalog.demo_catalog_temp.s3.secret-access-key', s3_secret_key)
    .config('spark.sql.catalog.demo_catalog_temp.client.region', 'us-east-1'))


Raw S3 reads/writes use Hadoop's S3A connector, which has its own configuration. Polaris Iceberg tables use S3FileIO above. Both reach the same SeaweedFS service.


In [ ]:
builder = (builder
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.endpoint', s3_endpoint)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .config('spark.hadoop.fs.s3a.aws.credentials.provider', 'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
    .config('spark.hadoop.fs.s3a.access.key', s3_access_key)
    .config('spark.hadoop.fs.s3a.secret.key', s3_secret_key))
spark = builder.getOrCreate()
spark.sparkContext.setLogLevel('WARN')
assert spark.sparkContext.master.startswith('spark://'), 'Expected the standalone Spark master'
print('Spark version:', spark.version)
print('Application ID:', spark.sparkContext.applicationId)
print('Spark master:', spark.sparkContext.master)
print('Spark UI:', spark.sparkContext.uiWebUrl)


## 5. Read the retained Polaris table

Read the three baseline rows and verify their IDs. All subsequent copies originate here. Do not insert extra rows into this prerequisite table.


In [ ]:
baseline_table = 'demo_catalog_temp.demo_database_temp.demo_table_temp'
source = spark.table(baseline_table)
source.orderBy('id').show(truncate=False)
assert [row.id for row in source.select('id').orderBy('id').collect()] == [1, 2, 3]
source.createOrReplaceTempView('source_rows_temp')
print('PASS: retained lesson 16 table has the expected three rows.')


Inspect the existing two append snapshots and their manifest lists.


In [ ]:
spark.sql(f'SELECT committed_at, snapshot_id, parent_id, operation, manifest_list FROM {baseline_table}.snapshots ORDER BY committed_at').show(truncate=False)


## 6. Write and read Parquet on Hadoop HDFS

Write a new run-specific directory. `errorifexists` preserves existing results instead of overwriting them.

First run the HDFS directory preparation commands in **17-0-Jupyter.md**. `/training/notebook-demo` must be writable by the `spark` account; do not change shared HDFS permissions or impersonate another user to bypass an access error.


In [ ]:
hdfs_raw = hdfs_root + '/raw-parquet-temp'
source.coalesce(1).write.mode('errorifexists').parquet(hdfs_raw)
hdfs_rows = spark.read.parquet(hdfs_raw)
hdfs_rows.orderBy('id').show()
assert hdfs_rows.count() == 3
print('HDFS Parquet directory:', hdfs_raw)


## 7. Register an ordinary Hive external table over HDFS

Create a new Hive database in the shared metastore. The namespace name is unique to this run. HiveServer2 is not involved.


In [ ]:
spark.sql(f"CREATE DATABASE spark_catalog.{namespace} LOCATION '{hdfs_root}/hive-database-temp'").show()
spark.sql(f'SHOW DATABASES IN spark_catalog').show(truncate=False)


Register the HDFS Parquet data as a Hive external table. The table is ordinary Hive/Parquet, not Iceberg. Its metastore entry points to the files written in section 6.


In [ ]:
hive_table = f'spark_catalog.{namespace}.hive_table_temp'
spark.sql(f"CREATE EXTERNAL TABLE {hive_table} (id BIGINT, product STRING, amount DECIMAL(10,2)) STORED AS PARQUET LOCATION '{hdfs_raw}'").show()
spark.table(hive_table).orderBy('id').show()
assert spark.table(hive_table).count() == 3


## 8. Create an Iceberg HadoopCatalog table on HDFS

This catalog stores its table metadata on HDFS without registering it in Polaris or Hive Metastore.


In [ ]:
spark.sql(f'CREATE NAMESPACE hadoop_iceberg.{namespace}').show()
hadoop_table = f'hadoop_iceberg.{namespace}.hadoop_table_temp'
spark.sql(f"CREATE TABLE {hadoop_table} USING iceberg TBLPROPERTIES ('format-version'='2') AS SELECT * FROM source_rows_temp").show()
spark.table(hadoop_table).orderBy('id').show()
assert spark.table(hadoop_table).count() == 3


## 9. Create an Iceberg HiveCatalog table on HDFS

HiveCatalog uses the Hive database already created above, but registers a different Iceberg table. Its metadata and Parquet data are on HDFS.


In [ ]:
hive_iceberg_table = f'hive_iceberg.{namespace}.hive_iceberg_table_temp'
spark.sql(f"CREATE TABLE {hive_iceberg_table} USING iceberg LOCATION '{hdfs_root}/hive-iceberg-table-temp' TBLPROPERTIES ('format-version'='2') AS SELECT * FROM source_rows_temp").show()
spark.table(hive_iceberg_table).orderBy('id').show()
assert spark.table(hive_iceberg_table).count() == 3


List tables through the shared Hive namespace. The ordinary Hive table and Iceberg table are both registered there. Use the appropriate catalog when querying each table.


In [ ]:
spark.sql(f'SHOW TABLES IN spark_catalog.{namespace}').show(truncate=False)


## 10. Write raw Parquet to SeaweedFS S3

This is direct object-store IO through `s3a://`, without an Iceberg catalog commit. It demonstrates why merely uploading Parquet files does not create a registered Iceberg table.


In [ ]:
s3_raw = s3a_root + '/raw-parquet-temp'
source.coalesce(1).write.mode('errorifexists').parquet(s3_raw)
s3_rows = spark.read.parquet(s3_raw)
s3_rows.orderBy('id').show()
assert s3_rows.count() == 3
print('Raw S3 Parquet prefix:', s3_raw)


## 11. Create a separate Polaris Iceberg table on S3

Create a new namespace inside the retained catalog. Its S3 path remains within lesson 16's allowed `s3://demo-catalog-temp/iceberg` root. The baseline namespace and table remain unchanged.

Let Polaris choose the namespace and table locations under its default warehouse layout. This deployment rejects arbitrary custom namespace locations even when they are within an allowed S3 root; no server feature changes are needed.


In [ ]:
spark.sql(f'CREATE NAMESPACE demo_catalog_temp.{namespace}').show()
polaris_table = f'demo_catalog_temp.{namespace}.polaris_table_temp'
spark.sql(f"CREATE TABLE {polaris_table} USING iceberg TBLPROPERTIES ('format-version'='2') AS SELECT * FROM source_rows_temp").show()
spark.table(polaris_table).orderBy('id').show()
assert spark.table(polaris_table).count() == 3


Read the generated Iceberg manifest records through metadata tables. Iceberg creates valid manifests during commits; do not fabricate JSON files and call them Iceberg manifests.


In [ ]:
spark.sql(f'SELECT snapshot_id, operation, manifest_list FROM {polaris_table}.snapshots').show(truncate=False)
spark.sql(f'SELECT path, added_data_files_count FROM {polaris_table}.manifests').show(truncate=False)
spark.sql(f'SELECT file_path, record_count FROM {polaris_table}.files').show(truncate=False)


Fetch the new table registration from Polaris, then read its current metadata JSON directly through SeaweedFS S3A. Compare the UUID and snapshot ID to prove the catalog and storage refer to the same table. Request a new token if the earlier one has expired.


In [ ]:
table_path = f'/v1/demo_catalog_temp/namespaces/{namespace}/tables/polaris_table_temp'
registered_response = requests.get(polaris_uri + table_path, headers=api_headers, timeout=30)
registered_response.raise_for_status()
registered = registered_response.json()
metadata_uri = registered['metadata-location']
metadata_lines = spark.read.text(metadata_uri.replace('s3://', 's3a://', 1)).collect()
stored_metadata = json.loads('\n'.join(row.value for row in metadata_lines))
assert stored_metadata['table-uuid'] == registered['metadata']['table-uuid']
assert stored_metadata['current-snapshot-id'] == registered['metadata']['current-snapshot-id']
print('PASS: Polaris registration and SeaweedFS metadata UUID/snapshot agree.')
print('Current metadata:', metadata_uri)


## 12. Reconcile data across all backends

Compare complete rows, not only counts. Both EXCEPT ALL directions must be empty; this catches missing rows, extra rows, and changed values.


In [ ]:
tables = {'Hive external / HDFS': hive_table, 'Iceberg HadoopCatalog / HDFS': hadoop_table,
          'Iceberg HiveCatalog / HDFS': hive_iceberg_table, 'Iceberg Polaris / S3': polaris_table}
for label, table in tables.items():
    actual = spark.table(table).select('id', 'product', 'amount')
    assert actual.exceptAll(source).count() == 0, label + ': unexpected rows'
    assert source.exceptAll(actual).count() == 0, label + ': missing rows'
    print('PASS:', label)
assert source.exceptAll(spark.read.parquet(hdfs_raw)).count() == 0
assert source.exceptAll(spark.read.parquet(s3_raw)).count() == 0
assert spark.table(baseline_table).count() == 3
print('PASS: baseline preserved and all four table backends reconcile.')


## 13. Record retained examples for later lessons

Save a small inventory under `/workspace/notebooks/runs`. It includes locations and names, never credentials or OAuth tokens. These resources remain in their respective catalogs and storage systems; stopping Spark does not delete them.


In [ ]:
inventory = {'run_id': run_id, 'namespace': namespace, 'baseline_table': baseline_table,
             'tables': tables, 'hdfs_root': hdfs_root, 's3_root': s3_root,
             'hdfs_parquet': hdfs_raw, 's3_parquet': s3_raw, 'metadata_uri': metadata_uri,
             'application_id': spark.sparkContext.applicationId, 'validation': 'PASS'}
run_dir = notebook_home / 'runs'
run_dir.mkdir(exist_ok=True)
inventory_file = run_dir / (run_id + '.json')
inventory_file.write_text(json.dumps(inventory, indent=2) + '\n')
print(json.dumps(inventory, indent=2))
print('Saved inventory:', inventory_file)


## 14. View Kubernetes logs from the VM terminal

These are **VM terminal commands**, not Python notebook cells. Run `okd-env` first. `oc logs` is the Kubernetes equivalent of `docker logs`; `-f` follows a stream and Ctrl+C stops following without stopping the workload.

Jupyter server/kernel lifecycle:

```bash
oc -n bigdata logs -f spark-master-0 -c jupyter --tail=50 --timestamps
```

Spark master scheduling:

```bash
oc -n bigdata logs spark-master-0 -c spark-master --since=10m --timestamps
```

Worker scheduling:

```bash
oc -n bigdata logs spark-worker-0 -c spark-worker --since=10m --timestamps
```

Polaris requests:

```bash
oc -n bigdata logs deployment/polaris -c polaris --since=10m --timestamps
```

Hive catalog requests:

```bash
oc -n bigdata logs deployment/hive-metastore --since=10m --timestamps
```

HDFS NameNode:

```bash
oc -n bigdata logs namenode-0 --since=10m --timestamps
```

S3 server:

```bash
oc -n bigdata logs seaweedfs-0 -c seaweedfs --since=10m --timestamps
```

Driver results and exceptions appear in notebook cell outputs. `oc logs` shows container stdout/stderr and may not include all kernel or executor task output. Use the Spark UI and notebook traceback as well. See **17-0-Jupyter.md** for browser forwards and saving directly in `/workspace/notebooks`.


## 15. Release Spark and keep the data

Stop the driver/executors when finished, then save the notebook in JupyterLab. **No tables, namespaces, buckets, or HDFS paths are dropped.** The run inventory identifies everything retained. To run this notebook again, restart the kernel and start from cell 1 to get a new run name.


In [ ]:
spark.stop()
print('PASS: integration notebook completed; Spark stopped; lesson 16 and this run remain available.')


References: [Spark Hive tables](https://spark.apache.org/docs/3.5.6/sql-data-sources-hive-tables.html), [Iceberg catalog configuration](https://iceberg.apache.org/docs/1.10.0/spark-configuration/), [Iceberg metadata queries](https://iceberg.apache.org/docs/1.10.0/spark-queries/), and [Polaris 1.7.0](https://polaris.apache.org/releases/1.7.0/).
